# R7.1 — ADT 独立参考验证结果审计及共同空间范围敏感性分析

**目标：复核 R7.0 的 1993–2023 ADT/GLORYS 验证；不改原始 ADT 跟踪算法、不重新训练、不重新标注真值。**

全程读取既有 R7.0 结果 ZIP，复核日期覆盖、追踪失败、六断面趋势和多交点；可选使用已保存在本地的流轴 NPZ 重算 **两条轴均截断到 144.3°E** 的空间误差。

此 Notebook **不能替代真实模型预测导出**。R6.5 缺失 `predictions_manifest.csv` 时，预测 vs ADT 一律保持 NOT EXECUTED。

In [ ]:
# ===== 0. 环境与路径 =====
from pathlib import Path
import os, io, zipfile, json, hashlib, traceback, warnings, time
import numpy as np, pandas as pd
from scipy.spatial import cKDTree
from IPython.display import display, FileLink
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ROOT = Path(os.environ.get('KUROSHIO_ROOT', '/mnt/d/KuroshioDaily1993_2025'))
# 若只下载了实验 ZIP，没有安装本地项目，也可在容器/电脑中配置 KUROSHIO_ROOT
if not ROOT.exists(): ROOT = Path('/mnt/data')
RUN = ROOT / 'R70_ADT_Independent_Python'
RESULTS_ZIP = ROOT / 'R70_PY_上传给ChatGPT_ADT独立参考验证完整结果.zip'
QC_ZIP = ROOT / 'R70_PY_上传给ChatGPT_日期覆盖与流轴追踪QC.zip'
# 用户可按实际文件位置修改这两项
if not RESULTS_ZIP.exists(): RESULTS_ZIP = Path('/mnt/data/R70_PY_上传给ChatGPT_ADT独立参考验证完整结果.zip')
if not QC_ZIP.exists(): QC_ZIP = Path('/mnt/data/R70_PY_上传给ChatGPT_日期覆盖与流轴追踪QC.zip')
ADT_AX = RUN/'adt_axes_npz'
GLORYS_AX = ROOT/'final/tracks_final'
OUTPUT = ROOT/'R71_ADT_验证审计'
REPORT=OUTPUT/'reports';FIG=OUTPUT/'figures';TABLES=OUTPUT/'tables'
for p in [OUTPUT,REPORT,FIG,TABLES]:p.mkdir(parents=True,exist_ok=True)
# 该抽样用于审计共域误差，不依赖误差挑选日期；全量选项不需要重追踪
PER_YEAR_SAMPLE = 24
SHARED_DOMAIN_ALL = False
EAST_LIMIT_E = 144.3
SAMPLE_SPACING_KM = 5.0
BOOTSTRAP_REPS = 800
np.random.seed(20261016)
print('ROOT:',ROOT,'RESULT ZIP:',RESULTS_ZIP.exists(),'QC ZIP:',QC_ZIP.exists())
assert RESULTS_ZIP.exists() and QC_ZIP.exists(), '找不到 R7.0 的两份 ZIP，请改为文件实际路径'

In [ ]:
# ===== 1. 校验 ZIP，载入真实报告 =====
def checksum(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1<<20),b''):h.update(block)
    return h.hexdigest()
def checkzip(path):
    with zipfile.ZipFile(path) as z:
        broken=z.testzip()
        if broken:raise RuntimeError('ZIP 内部 CRC 失败 '+broken)
    ss=Path(str(path)+'.sha256')
    ok=None
    if ss.exists():
        expected=ss.read_text(encoding='utf-8-sig').strip().split()[0].lower()
        ok=(checksum(path)==expected)
        if not ok:raise RuntimeError('外部 SHA256 不匹配：'+str(path))
    return dict(file=path.name,sha256=checksum(path),external_sha_match=ok)
zip_audit=[checkzip(RESULTS_ZIP),checkzip(QC_ZIP)]
with zipfile.ZipFile(RESULTS_ZIP) as z:
    path_daily=pd.read_csv(io.BytesIO(z.read('daily/每日ADT_vs_GLORYS完整流轴空间误差.csv')))
    daily_sections=pd.read_csv(io.BytesIO(z.read('daily/六断面逐日ADT与GLORYS成对误差.csv')))
    all_intersections=pd.read_csv(io.BytesIO(z.read('daily/每日六断面交点和多交点信息.csv')))
    monthly=pd.read_csv(io.BytesIO(z.read('monthly/六断面ADT统一基准月尺度距平.csv')))
    old_trends=pd.read_csv(io.BytesIO(z.read('tables/六断面长期趋势与95CI.csv')))
    pending_forecast=[i for i in z.namelist() if '预测vs独立ADT' in i]
    have_forecast=any(z.getinfo(i).file_size>4 for i in pending_forecast)
with zipfile.ZipFile(QC_ZIP) as z:
    qc=pd.read_csv(io.BytesIO(z.read('05_ADT逐日追踪及QC报告.csv')))
    manifest=pd.read_csv(io.BytesIO(z.read('01_ADT原始日期Manifest.csv')))
    state=json.loads(z.read('R70_PY_最终实验状态.json').decode('utf-8'))
print('SHA/CRC:',zip_audit,'FORECAST_EXECUTED?',have_forecast)
print('原始文件:',len(manifest),'QC:',len(qc),'paired days:',len(path_daily),'section pairs:',len(daily_sections),'monthly rows:',len(monthly))
assert len(manifest)==11322 and len(qc)==11322, '覆盖与历史结果不匹配，核对 ZIP 版本'
assert len(path_daily)==state['paired_monitoring_days'], '空间误差与状态 JSON 样本数不一致'
assert not have_forecast, '检测到非空预测文件，请单独审计预测起报日合同'
(TABLES/'01_ZIP完整性.csv').write_text(pd.DataFrame(zip_audit).to_csv(index=False),encoding='utf-8-sig')

In [ ]:
# ===== 2. 日期覆盖、连续失败与特定年份的审查（不静默剔除） =====
qc['date_dt']=pd.to_datetime(qc.date)
qc['year']=qc.date_dt.dt.year
qc['month']=qc.date_dt.dt.to_period('M').astype(str)
qcy=pd.crosstab(qc.year,qc.status)
qcy['total']=qcy.sum(axis=1)
qcy['success_rate']=qcy.get('success',0)/qcy['total']
qcy.to_csv(TABLES/'02_逐年ADT追踪成功率.csv',encoding='utf-8-sig')
qcm=pd.crosstab(qc.month,qc.status)
qcm['total']=qcm.sum(axis=1);qcm['success_rate']=qcm.get('success',0)/qcm.total
qcm.to_csv(TABLES/'03_逐月ADT追踪成功率.csv',encoding='utf-8-sig')
fail=(qc.sort_values('date_dt').status.to_numpy()!='success')
st=np.flatnonzero(np.diff(np.r_[False,fail,False].astype(int))==1)
en=np.flatnonzero(np.diff(np.r_[False,fail,False].astype(int))==-1)
qq=qc.sort_values('date_dt').reset_index(drop=True)
runs=pd.DataFrame([{'start':qq.loc[i,'date'],'end':qq.loc[j-1,'date'],'days':j-i} for i,j in zip(st,en)]).sort_values('days',ascending=False)
runs.to_csv(TABLES/'04_最长连续失败期.csv',index=False,encoding='utf-8-sig')
print('成功:',int((qc.status=='success').sum()),'失败:',int((qc.status!='success').sum()),'成功率:',round((qc.status=='success').mean()*100,2),'%')
print('2023成功:',qcy.loc[2023,'success'],'失败:',qcy.loc[2023,'qc_rejected'])
print('最长失败期：');display(runs.head(10));display(qcm.loc['2023-01':'2023-12'])

In [ ]:
# ===== 3. 原始报告真实数值及配对一致性审计 =====
common_dates=set(path_daily.date)
section_counts=daily_sections.groupby('section').agg(paired_days=('date','nunique'),bias_km=('diff_km','mean'),mae_km=('diff_km',lambda x:np.mean(np.abs(x))),rmse_km=('diff_km',lambda x:np.sqrt(np.mean(np.asarray(x)**2)))).reset_index()
section_counts.to_csv(TABLES/'05_断面逐日误差汇总.csv',index=False,encoding='utf-8-sig')
base=path_daily[['rmse_km','mae_km','p90_km','adt_terminal_lon','glorys_terminal_lon']].describe(percentiles=[.5,.9,.95]).T
base.to_csv(TABLES/'06_整体空间误差原始描述统计.csv',encoding='utf-8-sig')
print('整体 RMSE 均值',round(path_daily.rmse_km.mean(),3),'中位数',round(path_daily.rmse_km.median(),3))
print('ADT 终止经度均值',round(path_daily.adt_terminal_lon.mean(),3),'GLORYS',round(path_daily.glorys_terminal_lon.mean(),3))
print('断面：');display(old_trends);display(section_counts)
# 中途写盘的旧路径文件只有10200行，必须以最终10279行完整文件为权威
with zipfile.ZipFile(RESULTS_ZIP) as z:
    previous=pd.read_csv(io.BytesIO(z.read('daily/每日整体路径距离.csv')))
extra=sorted(set(path_daily.date)-set(previous.date))
(TABLES/'07_非权威中途缓存漏掉的日期.csv').write_text(pd.DataFrame({'date':extra}).to_csv(index=False),encoding='utf-8-sig')
print('旧增量缓存漏行数:',len(extra),'；使用 每日ADT_vs_GLORYS完整流轴空间误差.csv 为准')

In [ ]:
# ===== 4. 六断面鲁棒性：不含多交点的敏感性 / 1993-2022趋势 =====
def summarize_monthly(v,cutoff='2022-12'):
    rows=[]
    for sec,g in v[v.month<=cutoff].groupby('section'):
        if len(g)<24:continue
        dt=pd.PeriodIndex(g.month,freq='M').to_timestamp()
        x=(dt-dt.min()).days.to_numpy()/365.2425
        a=g.adt_km.to_numpy(float);b=g.glorys_km.to_numpy(float)
        sa=float(np.polyfit(x,a,1)[0]*10);sb=float(np.polyfit(x,b,1)[0]*10)
        rows.append(dict(section=sec,months=len(g),correlation=float(np.corrcoef(a,b)[0,1]),
                         adt_km_decade=sa,glorys_km_decade=sb,diff_km_decade=sb-sa,
                         monthly_bias_km=float((b-a).mean()),monthly_rmse_km=float(np.sqrt(np.mean((b-a)**2)))))
    return pd.DataFrame(rows)
robust_2022=summarize_monthly(monthly)
robust_2022.to_csv(TABLES/'08_1993至2022逐月趋势敏感性.csv',index=False,encoding='utf-8-sig')
unique= daily_sections[(daily_sections.adt_crossings==1)&(daily_sections.glorys_crossings==1)]
unique_count=unique.groupby('section').agg(n=('diff_km','size'),rmse=('diff_km',lambda x:np.sqrt(np.mean(x*x))),mean_bias=('diff_km','mean')).reset_index()
unique_count.to_csv(TABLES/'09_仅单交点断面敏感性.csv',index=False,encoding='utf-8-sig')
print('截至2022年末趋势:');display(robust_2022)
print('排除多交点的断面定位误差:');display(unique_count)

In [ ]:
# ===== 5. 可选共域重算：不重做 ADT 追踪，仅读取已有 NPZ =====
EARTH_RADIUS_KM=6371.0088
def xyz(k):
    q=np.deg2rad(np.asarray(k,float));L=q[:,0];P=q[:,1]
    return EARTH_RADIUS_KM*np.column_stack([np.cos(P)*np.cos(L),np.cos(P)*np.sin(L),np.sin(P)])
def gc_distance(chord):
    return 2*EARTH_RADIUS_KM*np.arcsin(np.clip(np.asarray(chord)/(2*EARTH_RADIUS_KM),0,1))
def seg_km(x):return gc_distance(np.linalg.norm(np.diff(xyz(x),axis=0),axis=1))
def densify(x,step=5):
    q=[x[0]]
    for a,b,d in zip(x[:-1],x[1:],seg_km(x)):
        if not np.isfinite(d) or d>250:raise ValueError('不连续轨迹 >250 km')
        n=max(1,int(np.ceil(d/step)))
        q.extend(a+(b-a)*(k/n) for k in range(1,n+1))
    return np.asarray(q)
def nearest_rmse(a,b,step=5):
    aa=densify(a,step);bb=densify(b,step)
    x=xyz(aa);y=xyz(bb)
    d=np.r_[gc_distance(cKDTree(y).query(x)[0]),gc_distance(cKDTree(x).query(y)[0])]
    return dict(rmse_km=float(np.sqrt(np.mean(d*d))),mae_km=float(np.mean(d)),p90_km=float(np.quantile(d,.9)))
def east_truncate(x,limit=144.3):
    # First eastward crossing of terminal longitude; no extrapolation.
    for k in range(1,len(x)):
        lo,hi=x[k-1,0],x[k,0]
        if lo < limit <= hi:
            frac=(limit-lo)/(hi-lo)
            return np.vstack([x[:k],x[k-1]+frac*(x[k]-x[k-1])])
    raise ValueError('轨迹未完成预定经度，不能外推')
def load_native(path,kind):
    with np.load(path,allow_pickle=False) as z:
        if kind=='ADT':a=np.column_stack([z['axis_lon'],z['axis_lat']])
        else:a=np.column_stack([z['lon'],z['lat']])
    a=np.asarray(a,float);valid=np.isfinite(a).all(axis=1)
    if not valid.all():raise ValueError('轨迹含 NaN：禁止跨断点连接')
    return a
if ADT_AX.is_dir() and GLORYS_AX.is_dir():
    rng=np.random.default_rng(20261016)
    p=path_daily.copy();p['year']=pd.to_datetime(p.date).dt.year
    selected=[]
    for year,g in p.groupby('year'):
        g=g.sort_values('date')
        if SHARED_DOMAIN_ALL: selected.extend(g.date.tolist())
        else:
            count=min(PER_YEAR_SAMPLE,len(g));ix=np.linspace(0,len(g)-1,count,dtype=int)
            selected.extend(g.iloc[ix].date.tolist())
    out=[]
    for date in selected:
        k=date.replace('-','')
        try:
            a=load_native(ADT_AX/(k+'.npz'),'ADT')
            b=load_native(GLORYS_AX/(k+'.npz'),'GLORYS')
            raw=nearest_rmse(a,b,SAMPLE_SPACING_KM)
            common=nearest_rmse(east_truncate(a,EAST_LIMIT_E),east_truncate(b,EAST_LIMIT_E),SAMPLE_SPACING_KM)
            out.append(dict(date=date,year=int(date[:4]),original_rmse_km=raw['rmse_km'],
                            common_rmse_km=common['rmse_km'],delta_km=common['rmse_km']-raw['rmse_km'],
                            original_mae_km=raw['mae_km'],common_mae_km=common['mae_km'],valid=True))
        except Exception as e:
            out.append(dict(date=date,year=int(date[:4]),valid=False,reason=str(e)))
    shared=pd.DataFrame(out)
    shared.to_csv(TABLES/'10_共同终止经度144.3E空间距离敏感性.csv',index=False,encoding='utf-8-sig')
    good=shared[shared.valid]
    print('共域重算：',len(good),'/',len(shared),'成功')
    if not good.empty:
        print(good[['original_rmse_km','common_rmse_km','delta_km']].agg(['mean','median']).round(3))
        display(good.groupby('year').agg(n=('date','size'),old=('original_rmse_km','mean'),common=('common_rmse_km','mean')).round(2))
else:
    shared=pd.DataFrame()
    print('未找到本机ADT/GLORYS NPZ轨迹目录，跳过共域重算。CSV统计审计仍然完整。')

In [ ]:
# ===== 6. 预览/输出审稿友好的质量检查图；不做成绩美化 =====
plt.rcParams.update({'font.size':10,'axes.grid':True,'grid.alpha':.17})
fig,ax=plt.subplots(figsize=(12,3.8),dpi=180)
c=qcm.copy();c.index=pd.to_datetime(c.index)
ax.plot(c.index,c.success_rate*100,color='#346782',lw=1.4)
ax.axhline(80,color='#bb704b',ls='--',lw=1,label='80% coverage threshold (diagnostic)')
ax.set(ylim=(0,105),ylabel='ADT reference track completion (%)',xlabel='Date',title='R7.0 reference coverage by month')
ax.legend();fig.tight_layout()
fig.savefig(FIG/'R71_Fig1_Monthly_ADT_Track_Coverage.png',dpi=250)
fig.savefig(FIG/'R71_Fig1_Monthly_ADT_Track_Coverage.pdf');plt.close(fig)
fig,ax=plt.subplots(figsize=(8.5,4.8),dpi=180)
w=0.32;loc=np.arange(len(old_trends))
ax.bar(loc-w/2,old_trends.adt_km_decade,w,label='ADT',color='#487998')
ax.bar(loc+w/2,old_trends.glorys_km_decade,w,label='GLORYS 50m',color='#c38554')
ax.set_xticks(loc,old_trends.section)
ax.set(ylabel='Linear slope (km / decade)',xlabel='Section',title='R7.0 monthly path displacement trends')
ax.axhline(0,color='.35',lw=.8);ax.legend();fig.tight_layout()
fig.savefig(FIG/'R71_Fig2_Six_Section_Trends.png',dpi=250)
fig.savefig(FIG/'R71_Fig2_Six_Section_Trends.pdf');plt.close(fig)
if not shared.empty and shared.valid.any():
    fig,ax=plt.subplots(figsize=(8.5,4.2))
    v=shared[shared.valid]
    ax.scatter(v.original_rmse_km,v.common_rmse_km,s=10,alpha=.4,color='#3e7897')
    mi=min(v.original_rmse_km.min(),v.common_rmse_km.min());mx=max(v.original_rmse_km.max(),v.common_rmse_km.max());ax.plot([mi,mx],[mi,mx],color='.4',ls='--')
    ax.set(xlabel='Original unequal-endpoint RMSE (km)',ylabel='Shared-extent RMSE (km)',title='Sensitivity to common downstream extent')
    fig.tight_layout();fig.savefig(FIG/'R71_Fig3_Shared_Domain_Sensitivity.png',dpi=250);fig.savefig(FIG/'R71_Fig3_Shared_Domain_Sensitivity.pdf');plt.close(fig)
print('图表输出位置：',FIG)

In [ ]:
# ===== 7. 不确定性与投稿边界自动记录，自动 ZIP + SHA256 =====
summary={
 'r70_checksums':zip_audit,
 'raw_adt_days':int(len(qc)),
 'accepted_adt_days':int((qc.status=='success').sum()),
 'rejected_adt_days':int((qc.status!='success').sum()),
 'accepted_2023_days':int(qcy.loc[2023,'success']),
 'longest_failure_period':runs.iloc[0].to_dict(),
 'paired_monitoring_days':int(len(path_daily)),
 'mean_daily_fullpath_rmse_km':float(path_daily.rmse_km.mean()),
 'median_daily_fullpath_rmse_km':float(path_daily.rmse_km.median()),
 'mean_adt_endpoint_lon':float(path_daily.adt_terminal_lon.mean()),
 'mean_glorys_endpoint_lon':float(path_daily.glorys_terminal_lon.mean()),
 'forecast_verification_performed':bool(have_forecast),
 'common_domain_sensitivity_n':int(shared.valid.sum()) if not shared.empty else 0,
 'comparison_limitations':['ADT 0.125deg surface front vs GLORYS 50m current core: definition mismatch',
                          'GLORYS may assimilate satellite altimetry: not fully statistically independent',
                          'ADT V6 Python port not numerically matched against MATLAB',
                          '2023 ADT tracker terminal failures: strong nonrandom missingness',
                          'R7.0 overall path metric has unmatched eastern endpoints',
                          'R7.0 saved prediction files empty; forecast-to-ADT comparison NOT executed',
                          'Never replace daily valid sample with test-target data or select dates by errors']}
(REPORT/'R71_结果审计状态.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
notes="""R7.1 REPORT / SCIENTIFIC LIMITATIONS
1. The reported 10,279 paired daily validation samples are monitoring axes only, not forecast errors.
2. 2023 is badly undercovered (91 of 365 accepted). August 20-December 31 missing continuously; the peak KE anomaly cannot be verified by these 2023 ADT axes.
3. Original 50.42 km average daily polyline RMSE includes differences in eastern spatial support. Report common-domain sensitivity alongside original result; do not silently substitute one with the other.
4. Trends use shared month/date pairs and an ADT-centered mean; A/B contain opposite-signed ADT/GLORYS trends, while D/E correlate strongly.
5. Missing months and rejected dates must remain explicit; changing the tracker after viewing GLORYS discrepancies requires new independent evaluation protocol.
6. The 2023 ADT failures merit an ADT-only recovery based on satellite field diagnostics, or independent satellite expert/manual adjudication, blinded to GLORYS prediction errors.
7. Forecast-to-ADT analysis needs actual saved forecast path positions, model checkpoint provenance, target dates, and valid causal training/validation splits.
"""
(REPORT/'R71_科研解释与实验边界.txt').write_text(notes,encoding='utf-8')
archive=ROOT/'R71_上传给ChatGPT_ADT结果复核及共域审计.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for item in sorted(OUTPUT.rglob('*')):
        if item.is_file():z.write(item,item.relative_to(OUTPUT))
h=checksum(archive)
Path(str(archive)+'.sha256').write_text(h+'  '+archive.name+'\n',encoding='utf-8')
with zipfile.ZipFile(archive) as z: assert z.testzip() is None
print('结果 ZIP:',archive,'SHA256:',h)
display(FileLink(str(archive)))